# 머신러닝 프로젝트

어떤 구역의 위치, 소득, 주택, 인구 정보가 주어졌다고 하자.

> **이 정보만으로 그 구역의 중위 주택가격을 예측할 수 있을까?**

이번 장에서는 California Housing 데이터를 이용해 이 질문을 따라간다.

중요한 것은 특정 알고리즘을 자세히 배우는 것이 아니다.  
머신러닝 프로젝트가 실제로 어떤 순서로 진행되는지 이해하는 것이 목적이다.

> **문제 확인 → 데이터 이해 → 훈련셋과 테스트셋 분리 → 훈련 데이터 탐색 → 데이터 준비 → 모델 훈련 → 평가**

> **원자료**
>
> - 강의내용: [42H: 머신러닝 — 2. 머신러닝 프로젝트](https://codingalzi.github.io/code-workout-ml/end2end-ml-project/)
> - 코드: [code-end2end_ml_project.ipynb](https://github.com/codingalzi/code-workout-ml/blob/master/notebooks/code-end2end_ml_project.ipynb)
>
> 이 노트는 위 자료에서 핵심 내용과 코드를 선별하여 강의용으로 재구성하였다.  
> 코드는 필요할 때 펼쳐볼 수 있도록 기본적으로 숨겨 두었다.

## 무엇을 예측하려는가?

1990년 미국 캘리포니아의 **20,640개 구역**에 대해 다음과 같은 정보가 조사되었다.

![California Housing 원자료](https://codingalzi.github.io/code-workout-ml/build/393384fca2314d9967c663f11aa9b5f9.png)

구역마다 경도, 위도, 주택 연령, 방 수, 인구, 가구 수, 중위소득, 해안 근접도 등이 기록되어 있다.

이 가운데 우리가 예측하려는 값은 **중위 주택가격**(`median_house_value`)이다.

따라서 이 문제는

- 타깃이 주어진 **지도학습**
- 연속적인 수치를 예측하는 **회귀**
- 여러 특성으로 하나의 값을 예측하는 **다중 회귀**

문제다.

### 데이터를 불러온다

원본 노트북에서는 `load_housing_data()` 함수를 이용해 데이터를 내려받고 판다스 데이터프레임으로 불러온다.

코드 자체보다 중요한 것은 그 다음 질문이다.

> **불러온 데이터가 실제로 어떤 모습인지 먼저 확인했는가?**

In [ ]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
    with tarfile.open(tarball_path) as housing_tarball:
        housing_tarball.extractall(path="datasets", filter="data")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing_full = load_housing_data()

## 모델보다 먼저 데이터를 본다

처음 몇 행과 데이터 구조를 확인하면 바로 몇 가지 사실을 알 수 있다.

- 전체 구역 수는 20,640개다.
- `ocean_proximity`는 범주형 특성이다.
- 나머지는 수치형 특성이다.
- `total_bedrooms`에는 결측치가 있다.

즉 아직 모델을 하나도 만들지 않았지만 벌써 해야 할 일이 보인다.

> **문자열 특성은 어떻게 처리할까? 결측치는 어떻게 처리할까?**

In [ ]:
housing_full.head()

In [ ]:
housing_full.info()

### 범주형 특성도 살펴본다

`ocean_proximity`는 다섯 범주로 구성된다.

- `<1H OCEAN`
- `INLAND`
- `NEAR OCEAN`
- `NEAR BAY`
- `ISLAND`

범주별 샘플 수가 크게 다르다는 점도 확인할 수 있다.

In [ ]:
housing_full["ocean_proximity"].value_counts()

## 수치만 보지 말고 분포를 본다

수치형 특성을 히스토그램으로 보면 표만 볼 때보다 훨씬 많은 것이 보인다.

![California Housing 특성별 히스토그램](https://codingalzi.github.io/code-workout-ml/build/d49c2d886c5b94677ec72362d67a68ab.png)

특히 다음 세 가지를 눈여겨본다.

1. 특성마다 **단위와 스케일이 다르다.**
2. `total_rooms`, `population` 등은 한쪽으로 많이 치우쳐 있다.
3. `housing_median_age`, `median_house_value`는 특정 상한에서 잘린 것처럼 보인다.

여기서 중요한 질문이 생긴다.

> **데이터가 이렇게 생겼다면 모델에 그대로 넣어도 될까?**

아직은 답하지 않는다. 먼저 평가용 데이터를 따로 남겨 둔다.

## 테스트셋은 먼저 남겨 둔다

모델을 만든 뒤 새로운 데이터에서도 잘 작동하는지 확인하려면  
훈련 과정에서 사용하지 않은 **테스트셋**이 필요하다.

원자료에서는 전체 데이터의 약 20%를 테스트셋으로 남긴다.

하지만 무작위로만 나누면 중요한 집단의 비율이 달라질 수 있다.  
California Housing에서는 **중위소득 구간**을 기준으로 계층 샘플링을 사용한다.

![중위소득 구간의 분포](https://codingalzi.github.io/code-workout-ml/build/631e04cc421a23692b32a9f655637c2b.png)

핵심은 함수 사용법이 아니다.

> **훈련셋과 테스트셋이 원래 데이터의 중요한 구성을 비슷하게 유지하도록 나눈다.**

In [ ]:
from sklearn.model_selection import train_test_split

housing_full["income_cat"] = pd.cut(housing_full["median_income"],
                                   bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                                   labels=[1, 2, 3, 4, 5])

strat_train_set, strat_test_set = train_test_split(
    housing_full, test_size=0.2,
    stratify=housing_full["income_cat"],
    random_state=42)

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

## 이제 훈련셋만 자세히 본다

테스트셋을 만든 뒤에는 본격적인 탐색을 **훈련셋만을 대상으로** 진행한다.

왜 그럴까?

테스트셋은 앞으로 만날 새로운 데이터를 대신한다.  
모델과 분석 방법을 결정할 때 테스트셋을 계속 들여다보면  
그 정보가 선택 과정에 들어가게 된다.

따라서 다음부터는 훈련셋을 복사해 탐색한다.

In [ ]:
housing = strat_train_set.copy()

## 위치와 가격을 함께 보면 무엇이 보일까?

경도와 위도를 이용해 구역을 지도처럼 표시하면  
주택가격의 지역적 차이를 한눈에 볼 수 있다.

![California Housing 지리적 분포](https://codingalzi.github.io/code-workout-ml/build/ff9f4f7576804adf683408c79ea347ba.png)

그림에서

- 원의 크기는 인구와 관련되고
- 색은 중위 주택가격을 나타낸다.

높은 가격의 구역이 완전히 무작위로 흩어져 있지는 않다.

> **위치가 주택가격 예측에 중요한 정보일 가능성이 있다.**

이제 다른 특성도 주택가격과 어떤 관계를 갖는지 살펴본다.

## 어떤 특성이 가격과 관련될까?

수치형 특성 사이의 **피어슨 상관계수**를 계산하면  
중위 주택가격과 중위소득의 선형 관계가 특히 강하게 나타난다.

![특성 사이의 상관관계](https://codingalzi.github.io/code-workout-ml/build/2013ce08faa4422e2df82a062c0b6052.png)

하지만 상관계수 하나만으로는 관계의 모양을 알기 어렵다.

그래서 산점도를 다시 본다.

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

### 중위소득과 주택가격

![중위소득과 중위 주택가격](https://codingalzi.github.io/code-workout-ml/build/f039151ecaba1102c35db4bad602b2c3.png)

전반적으로 중위소득이 높을수록 중위 주택가격도 높아지는 경향이 보인다.

동시에 다음도 보인다.

- 같은 소득 수준에서도 가격이 넓게 퍼져 있다.
- 약 50만 달러에서 가격이 잘려 있다.

즉

> **중위소득은 유용한 특성이지만 이것 하나만으로 가격을 설명할 수는 없다.**

그리고 상관관계가 높다는 사실만으로 인과관계를 주장할 수도 없다.

## 이제 모델에 넣을 데이터를 만든다

훈련셋에서 타깃을 분리한다.

- `housing`: 주택가격을 제외한 입력 특성
- `housing_labels`: 모델이 맞혀야 하는 주택가격

이제부터의 질문은

> **모델이 이 데이터를 실제로 사용할 수 있는 형태로 어떻게 바꿀 것인가?**

이다.

In [ ]:
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

## 첫 번째 문제: 결측치

`total_bedrooms`에는 값이 빠진 구역이 있다.

원자료에서는 세 가지 선택지를 소개한다.

1. 결측치가 있는 샘플을 삭제한다.
2. 해당 특성 자체를 삭제한다.
3. 특정 값으로 결측치를 채운다.

이 예제에서는 **중위수**로 채우는 방법을 사용하고  
`SimpleImputer` 변환기를 활용한다.

![SimpleImputer 적용 결과](https://codingalzi.github.io/code-workout-ml/build/78aafdeb2c6fddcbf2c53945138d85d6.png)

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
housing_num = housing.select_dtypes(include=[np.number])
X = imputer.fit_transform(housing_num)

## 두 번째 문제: 문자열 범주

`ocean_proximity`는 문자열이다.

머신러닝 모델이 이 값을 그대로 사용하기보다는  
각 범주를 별도의 수치 특성으로 표현하는 **원-핫 인코딩**을 적용한다.

![원-핫 인코딩](https://codingalzi.github.io/code-workout-ml/build/409e5077cd12e5441143f416f9f8834f.png)

예를 들어 `INLAND`는 하나의 숫자 2처럼 바꾸는 것이 아니라

```text
[0, 1, 0, 0, 0]
```

처럼 표현한다.

그렇게 해야 범주 사이에 존재하지 않는 크기 순서를 모델이 오해하지 않는다.

## 세 번째 문제: 수치형 특성의 스케일

특성마다 값의 범위가 크게 다르다.

예를 들어

- 중위소득은 한 자리 또는 두 자리 수
- 인구와 방 수는 수천 또는 수만 단위

일 수 있다.

원자료에서는 수치형 특성에 **표준화**를 적용하고,  
결측치 처리와 함께 하나의 파이프라인으로 묶는다.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler())
])

## 특성마다 다른 처리가 필요하다

수치형 특성과 범주형 특성은 같은 방법으로 처리할 수 없다.

그래서

- 수치형 특성 → 결측치 처리 + 표준화
- 범주형 특성 → 결측치 처리 + 원-핫 인코딩

을 각각 적용한 뒤 하나의 데이터로 합친다.

이 역할을 `ColumnTransformer`가 수행한다.

> **중요한 것은 코드를 외우는 것이 아니라  
> 어떤 종류의 데이터에 어떤 처리가 필요한지 이해하는 것이다.**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

num_attribs = ["longitude", "latitude", "housing_median_age", "total_rooms",
               "total_bedrooms", "population", "households", "median_income"]
cat_attribs = ["ocean_proximity"]

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

## 파이프라인은 왜 필요한가?

데이터를 준비하는 단계가 여러 개라면  
훈련 데이터와 새로운 데이터에 **항상 같은 순서의 처리**를 적용해야 한다.

파이프라인은

> **전처리 → 모델 훈련 → 예측**

과정을 하나의 객체처럼 연결해 준다.

이 장에서는 알고리즘 내부를 자세히 배우지 않고  
세 가지 회귀 모델을 비교한다.

- 선형회귀
- 결정트리 회귀
- 랜덤 포레스트 회귀

## 첫 모델: 선형회귀

선형회귀는 비교 기준이 되는 단순한 모델로 사용할 수 있다.

원자료에서는 전처리 과정과 선형회귀를 하나의 파이프라인으로 묶는다.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)

### RMSE로 예측 오차를 확인한다

회귀 모델의 성능은 **RMSE**로 평가한다.

RMSE는 예측값과 실제값의 차이를 하나의 숫자로 요약하며  
**0에 가까울수록 좋다.**

선형회귀의 훈련셋 RMSE는 약 6만 9천 정도다.

이 숫자만 보고 끝내지 않는다.

> **더 복잡한 모델은 더 잘할까?**

In [ ]:
from sklearn.metrics import root_mean_squared_error

housing_predictions = lin_reg.predict(housing)
lin_rmse = root_mean_squared_error(housing_labels, housing_predictions)
lin_rmse

## 결정트리는 훈련 데이터를 완벽하게 맞힌다

결정트리 회귀 모델을 훈련하면  
훈련셋 RMSE가 **0**이 나온다.

완벽한 모델을 찾은 것처럼 보인다.

정말 그럴까?

원자료의 해석은 반대다.

> **훈련 데이터에 지나치게 맞춰진 과대적합을 의심해야 한다.**

1장에서 배운 일반화 문제가 실제 프로젝트에서 등장한 것이다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing, housing_labels)

housing_predictions = tree_reg.predict(housing)
tree_rmse = root_mean_squared_error(housing_labels, housing_predictions)
tree_rmse

## 훈련 성능만으로 모델을 고르면 안 된다

결정트리는 훈련셋에서는 오차가 0이지만  
새로운 데이터에서도 그렇게 잘할 가능성은 낮다.

그렇다고 지금 테스트셋을 열어 보면 될까?

아직 아니다.

모델을 비교하고 선택하는 과정에서 테스트셋을 반복해서 사용하면  
테스트셋도 사실상 선택 과정의 일부가 된다.

그래서 **교차검증**을 사용한다.

## 교차검증: 훈련셋 안에서 여러 번 확인하기

![교차검증의 기본 아이디어](https://codingalzi.github.io/code-workout-ml/build/f1de2eb5855311bdbb48e07d7d4d48a8.png)

k-겹 교차검증은 훈련셋을 여러 부분으로 나눈 뒤

1. 한 부분은 검증에 사용하고
2. 나머지는 훈련에 사용하며
3. 검증에 사용할 부분을 바꾸어 여러 번 반복한다.

이렇게 하면 테스트셋을 건드리지 않고도  
모델의 일반화 성능을 비교할 수 있다.

In [ ]:
from sklearn.model_selection import cross_val_score

tree_rmses = -cross_val_score(tree_reg, housing, housing_labels,
                              scoring="neg_root_mean_squared_error",
                              cv=10)

## 세 모델을 비교하면

원자료의 10-겹 교차검증 결과는 대략 다음과 같다.

| 모델 | 평균 RMSE | 해석 |
|---|---:|---|
| 선형회귀 | 약 70,000 | 세 모델 중 오차가 가장 큼 |
| 결정트리 | 약 67,000 | 훈련 RMSE 0과 달리 검증 성능은 나쁨 |
| 랜덤 포레스트 | 약 47,000 | 세 모델 중 가장 좋은 성능 |

여기서 두 가지를 배울 수 있다.

첫째,

> **훈련 데이터에서 가장 잘 맞는 모델이 가장 좋은 모델은 아니다.**

둘째,

> **모델 비교는 새로운 데이터에서의 성능을 예상할 수 있는 방식으로 해야 한다.**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing,
                           RandomForestRegressor(random_state=42))

forest_rmses = -cross_val_score(forest_reg, housing, housing_labels,
                                scoring="neg_root_mean_squared_error",
                                cv=10)

pd.Series(forest_rmses).describe()

## 테스트셋은 언제 사용하는가?

원자료에서는 이후 랜덤 포레스트의 하이퍼파라미터를  
그리드 탐색과 랜덤 탐색으로 조정한 뒤 **마지막에 테스트셋**을 사용한다.

이 장에서는 미세 조정 방법을 자세히 다루지 않는다.  
여기서는 원칙만 기억한다.

> **훈련과 모델 선택이 모두 끝난 뒤 테스트셋으로 최종 성능을 확인한다.**

그리고 테스트셋을 전처리할 때도  
테스트셋에서 새로운 평균이나 중위수 등을 다시 계산하지 않는다.

훈련 과정에서 구한 값을 그대로 사용해야 한다.

이 원칙은 **데이터 누수**를 막는 데 중요하다.

## 처음 질문으로 돌아가 보자

처음에는 단순해 보였다.

> **“구역 정보를 이용해 주택가격을 예측하면 된다.”**

하지만 실제로는 그 전에 많은 판단이 필요했다.

- 어떤 값이 타깃인가?
- 데이터 구조는 어떤가?
- 결측치는 있는가?
- 범주형 특성은 있는가?
- 테스트셋을 어떻게 남길 것인가?
- 훈련 데이터에서 어떤 관계가 보이는가?
- 어떤 전처리가 필요한가?
- 훈련 성능을 그대로 믿어도 되는가?
- 모델을 어떤 방식으로 비교할 것인가?

그래서 머신러닝 프로젝트는 단순히

> **모델 선택 → `fit()` → 점수 확인**

으로 끝나는 일이 아니다.

## 이 장의 핵심

이 장에서 기억해야 할 흐름은 다음과 같다.

> **문제를 확인한다**  
> ↓  
> **데이터를 이해한다**  
> ↓  
> **테스트셋을 남긴다**  
> ↓  
> **훈련셋을 탐색한다**  
> ↓  
> **데이터를 정제하고 전처리한다**  
> ↓  
> **모델을 훈련한다**  
> ↓  
> **교차검증으로 모델을 비교한다**  
> ↓  
> **마지막에 테스트셋으로 평가한다**

특정 함수나 코드보다  
**왜 이 순서가 필요한지 설명할 수 있는 것**이 더 중요하다.

## 확인 문제

1. 데이터를 불러온 직후 모델부터 훈련하면 안 되는 이유는 무엇인가?
2. `total_bedrooms`와 `ocean_proximity`는 각각 어떤 전처리가 필요한가?
3. 전체 데이터를 자세히 탐색하기 전에 테스트셋을 남겨 두는 이유는 무엇인가?
4. 중위소득을 기준으로 계층 샘플링을 사용하는 이유는 무엇인가?
5. 결정트리의 훈련 RMSE가 0이라는 사실을 왜 좋은 성능의 증거로 볼 수 없는가?
6. 테스트셋 대신 교차검증을 이용해 모델을 비교하는 이유는 무엇인가?
7. 테스트셋을 전처리할 때 테스트셋 자체의 평균이나 중위수를 다시 계산하면 왜 문제가 되는가?

<details>
<summary><strong>핵심 답변 보기</strong></summary>

1. 데이터의 구조, 결측치, 범주형 특성, 분포 등의 문제를 먼저 파악해야 적절한 분석과 전처리를 설계할 수 있기 때문이다.
2. `total_bedrooms`의 결측치는 중위수 등으로 대체할 수 있고, `ocean_proximity`는 원-핫 인코딩으로 수치화할 수 있다.
3. 테스트셋이 모델과 분석 방법 선택에 영향을 받지 않은 새로운 데이터를 대표하도록 하기 위해서다.
4. 주택가격과 관련성이 높은 중위소득의 구간별 비율을 훈련셋과 테스트셋에서 비슷하게 유지하기 위해서다.
5. 훈련 데이터에 과대적합되었을 가능성이 매우 높기 때문이다.
6. 테스트셋을 모델 선택에 사용하지 않고도 훈련 데이터 안에서 일반화 성능을 비교하기 위해서다.
7. 테스트셋의 정보가 훈련 과정에 들어가는 데이터 누수가 발생하기 때문이다.

</details>